##### Setup
- Explore which factors (columns) are associated with `cancel_yn` (churn), using the full cleaned dataset (no train/test split needed for this exploratory step)

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)

CLEAN = "read_parquet('df_clean_part*.parquet')"

##### Baseline Churn Rate
- Reference point for comparing segment-level churn rates below

In [2]:
baseline = duckdb.sql(f"""
    SELECT round(count(*) FILTER (WHERE cancel_yn = '해지') * 100.0 / count(*), 4) AS baseline_churn_rate_pct
    FROM {CLEAN}
""").df()
baseline

,baseline_churn_rate_pct
0,4.0343


##### Categorical Columns: Churn Rate by Category
- For every categorical column, compute the churn rate (%) within each category value
- `deviation_from_baseline` shows how far each segment's churn rate is from the overall average — sorted by absolute deviation, so the strongest factors surface first
- Filters out tiny groups (`n < 1000`) to avoid noisy small-sample rates

In [3]:
categorical_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "AGMT_END_SEG", "BUNDLE_YN", "DIGITAL_GIGA_YN", "DIGITAL_ALOG_YN", "CH_LAST_DAYS_BF_GRP",
    "VOC_TOTAL_MONTH1_YN", "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM",
    "SMS_SEND_CLS_NM", "NFX_USE_YN", "YTB_USE_YN", "CH_FAV_RNK1",
]
# SQL에 들어가기 위해 여러 컬럼을 조회할 때 대괄호 없이 쉼표로 연결된 일반 텍스트
cols_sql = ", ".join(categorical_cols)
baseline_rate = baseline['baseline_churn_rate_pct'][0]

categorical_churn_scan = duckdb.sql(f"""
    WITH base AS (
        SELECT
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_cancelled,
            {cols_sql}
        FROM {CLEAN}
    ),
    unpvt AS (
        UNPIVOT base
        ON {cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        value,
        count(*) AS n,
        round(avg(is_cancelled) * 100, 4) AS churn_rate_pct,
        round(avg(is_cancelled) * 100 - {baseline_rate}, 4) AS deviation_from_baseline,
        round(avg(is_cancelled) * 100 / {baseline_rate}, 2) AS ratio_to_baseline
    FROM unpvt
    GROUP BY column_name, value
    HAVING count(*) >= 1000
    ORDER BY abs(round(avg(is_cancelled) * 100 - {baseline_rate}, 4)) DESC
""").df()

categorical_churn_scan.head(30)

,column_name,value,n,churn_rate_pct,deviation_from_baseline,ratio_to_baseline
0,AGMT_END_SEG,약정만료전 1개월,181220,17.6548,13.6205,4.38
1,AGMT_END_SEG,약정만료전 1~2개월,205110,14.8886,10.8543,3.69
2,VOC_STOP_CANCEL_MONTH1_YN,Y,464918,13.8674,9.8331,3.44
3,AGMT_END_SEG,약정만료전 2~3개월,224681,12.0250,7.9907,2.98
4,AGE_GRP10,20대,185727,11.7560,7.7217,2.91
5,AGMT_END_SEG,약정만료후 1개월~2개월,128414,9.4686,5.4343,2.35
6,AGMT_END_SEG,약정만료 1개월,328269,8.9935,4.9592,2.23
7,PROD_NM_GRP,세이버,1652,8.4746,4.4403,2.10
8,AGMT_END_SEG,약정만료후 2개월~3개월,118056,8.3232,4.2889,2.06
9,AGMT_END_SEG,약정만료전 3~6개월,795058,8.2893,4.2550,2.05


##### Finding

`AGMT_END_SEG` shows up 8 times in the top 30, so we should check for multicollinearity before treating these as 8 separate signals.

A disproportionate share of cancellations come from customers in their 20s — this age group could be a good target for retention marketing.

`VOC_STOP_CANCEL_MONTH1_YN` isn't very useful on its own, since it only flags customers who've already decided to cancel. It'd be more valuable to look at why some customers who call about cancelling end up *not* cancelling.

##### Numeric Columns: Average Value by cancel_yn
- Compare the average of each numeric column between the `해지`(cancelled) and `유지`(retained) groups
- A large gap between the two rows suggests that column is informative for churn prediction

In [4]:
numeric_cols = [
    "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB", "DIGITAL_SCRB",
    "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "TV_I_CNT",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM",
    "KIDS_USE_PV_MONTH1", "INHOME_RATE",
]

parts = []
for c in numeric_cols:
    expr = "round(avg(" + c + "), 3) AS avg_" + c
    parts.append(expr)

avg_exprs = ", ".join(parts)


numeric_churn_scan = duckdb.sql(f"""
    SELECT cancel_yn, count(*) AS n, {avg_exprs}
    FROM {CLEAN}
    GROUP BY cancel_yn
""").df()

numeric_churn_scan.T

,0,1
cancel_yn,유지,해지
n,21163859,889705
avg_TOTAL_USED_DAYS,2777.906,2300.376
avg_TV_SCRB,1.782,1.716
avg_ANALOG_SCRB,0.021,0.016
avg_DIGITAL_SCRB,1.761,1.7
avg_TOTAL_INTERNET_SCRB,0.489,0.656
avg_GIGA_INTERNET_SCRB,0.129,0.249
avg_TV_I_CNT,2.271,2.371
avg_CH_HH_AVG_MONTH1,4.613,3.218


# Problem Identification
- Every customer-month with `cancel_yn = '해지'` is counted as a cancellation, so the churn rate above assumes a cancelled customer's records end when they cancel
- **Question:** Do customers disappear from the data right after cancelling (the next month), or do their records continue for some time?
- If records continue, the same cancellation may be counted again in every later month, which would overstate churn rates and distort the segment comparison

##### Do Customers Still Appear After Their First `해지` Month?
- For each customer's first `해지` month, check whether they have any later row (`p_mt` after that month)
- If almost nobody appears afterward, that confirms `해지` = "left the dataset," so each churning customer contributes only one `해지` row out of their whole history

In [5]:
after_cancel_check = duckdb.sql(f"""
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    joined AS (
        SELECT
            fc.sha2_hash,
            fc.first_cancel_pmt,
            count(*) FILTER (WHERE c.p_mt > fc.first_cancel_pmt) AS n_rows_after
        FROM first_cancel fc
        JOIN {CLEAN} c ON c.sha2_hash = fc.sha2_hash
        GROUP BY fc.sha2_hash, fc.first_cancel_pmt
    )
    SELECT
        count(*) AS n_customers_with_cancel,
        count(*) FILTER (WHERE n_rows_after = 0) AS n_no_rows_after,
        count(*) FILTER (WHERE n_rows_after > 0) AS n_has_rows_after,
        round(count(*) FILTER (WHERE n_rows_after = 0) * 100.0 / count(*), 2) AS pct_no_rows_after
    FROM joined
""").df()

after_cancel_check

,n_customers_with_cancel,n_no_rows_after,n_has_rows_after,pct_no_rows_after
0,150613,15900,134713,10.56


##### What Does cancel_yn Look Like in the Rows After First `해지`?
- For the 134,713 customers who have rows after their first `해지`, check whether those later rows are mostly `유지` (reverted / reactivated) or mostly `해지` (status persists)

In [6]:
rows_after_status = duckdb.sql(f"""
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    rows_after AS (
        SELECT c.cancel_yn
        FROM {CLEAN} c
        JOIN first_cancel fc ON c.sha2_hash = fc.sha2_hash
        WHERE c.p_mt > fc.first_cancel_pmt
    )
    SELECT
        cancel_yn,
        count(*) AS n,
        round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS pct
    FROM rows_after
    GROUP BY cancel_yn
""").df()

rows_after_status

,cancel_yn,n,pct
0,해지,739092,100.0


##### Do the Other Columns Also Freeze After First `해지`?
- We already know `cancel_yn` stays `해지` in every row after the first cancellation. Do the other columns (`TV_I_CNT`, `CH_HH_AVG_MONTH1`, etc.) also stay frozen at the same value, or do they keep updating month to month?
- For customers with 2+ rows after their first `해지`, check how many distinct values each column takes across those rows — `n_distinct=1` means frozen

In [7]:
other_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "INHOME_RATE", "AGMT_END_SEG", "AGMT_END_YMD", "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB",
    "DIGITAL_SCRB", "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "BUNDLE_YN", "DIGITAL_GIGA_YN",
    "DIGITAL_ALOG_YN", "TV_I_CNT", "CH_LAST_DAYS_BF_GRP", "VOC_TOTAL_MONTH1_YN",
    "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM", "SMS_SEND_CLS_NM",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM", "CH_FAV_RNK1",
    "KIDS_USE_PV_MONTH1", "NFX_USE_YN", "YTB_USE_YN",
]
# INHOME_RATE/CH_25_RATIO_MONTH1/etc. are DOUBLE while the rest are VARCHAR in the
# cleaned data — UNPIVOT needs matching types, so cast everything to VARCHAR first
other_cols_sql = ", ".join(other_cols)
cast_cols_sql = ", ".join(f"CAST({c} AS VARCHAR) AS {c}" for c in other_cols)

frozen_check = duckdb.sql(f"""
    WITH raw AS (
        SELECT sha2_hash, p_mt, cancel_yn, {cast_cols_sql}
        FROM {CLEAN}
    ),
    first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM raw
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    rows_after AS (
        SELECT r.*
        FROM raw r
        JOIN first_cancel fc ON r.sha2_hash = fc.sha2_hash
        WHERE r.p_mt > fc.first_cancel_pmt
    ),
    customers_with_2plus AS (
        SELECT sha2_hash
        FROM rows_after
        GROUP BY sha2_hash
        HAVING count(*) >= 2
    ),
    unpvt AS (
        UNPIVOT (
            SELECT sha2_hash, {other_cols_sql}
            FROM rows_after
            WHERE sha2_hash IN (SELECT sha2_hash FROM customers_with_2plus)
        )
        ON {other_cols_sql}
        INTO
            NAME column_name
            VALUE value
    ),
    per_customer_col AS (
        SELECT sha2_hash, column_name, count(DISTINCT value) AS n_distinct
        FROM unpvt
        GROUP BY sha2_hash, column_name
    )
    SELECT
        column_name,
        count(*) AS n_customers,
        round(avg(CASE WHEN n_distinct = 1 THEN 100.0 ELSE 0 END), 2) AS pct_frozen
    FROM per_customer_col
    GROUP BY column_name
    ORDER BY pct_frozen
""").df()

frozen_check

,column_name,n_customers,pct_frozen
0,TOTAL_USED_DAYS,120987,0.00
1,CH_HH_AVG_MONTH1,120987,23.96
2,CH_25_RATIO_MEAN_3MM,120987,29.74
3,CH_25_RATIO_MONTH1,120987,29.74
4,VOC_TOTAL_MONTH1_YN,120987,33.62
5,INHOME_RATE,95899,41.10
6,CH_FAV_RNK1,120987,49.16
7,AGMT_END_SEG,120987,59.83
8,CH_LAST_DAYS_BF_GRP,120987,60.59
9,VOC_STOP_CANCEL_MONTH1_YN,120987,64.33


---

## Part 2. Re-analysis at Customer Level (One Row per Customer)

##### Why One Row per Customer
- Churned customers are recorded as `해지` in every month they appear (`538e72fa`: 739,092 later rows, all `해지`), so counting rows counts the same churner many times
- Churned customers have 1 row each after the fix but active customers appear ~10 times each, so a row-level rate also mixes the two groups unevenly
- Fix: keep one row per customer — the first month they appear — as the in-memory table `churn_base` (no new file is written)
- The cell below builds it once; run it first whenever the kernel restarts, then any of Parts 2–4 can run on its own
- From here on, churn rate means the **share of customers who are churners** (customer-level)

In [8]:
import duckdb
import pandas as pd

# Run this cell once (e.g. after a kernel restart); Parts 2-4 all read churn_base
pd.set_option('display.max_rows', None)
CLEAN = "read_parquet('df_clean_part*.parquet')"

# One row per customer: the first month each customer appears.
# Churners are recorded as 해지 in every month they appear (Part 5), so this keeps exactly one row each.
# TABLE (not VIEW): computed once and kept in memory, so later queries don't redo the window function
duckdb.sql(f"""
    CREATE OR REPLACE TABLE churn_base AS
    SELECT * EXCLUDE (rn)
    FROM (
        SELECT *, row_number() OVER (PARTITION BY sha2_hash ORDER BY p_mt) AS rn
        FROM {CLEAN}
    )
    WHERE rn = 1
""")

In [9]:
# Sanity check: 2,135,261 customers, 150,613 churners (one row each) 
truncate_check = duckdb.sql("""
    SELECT
        count(*) AS n_rows,
        count(*) FILTER (WHERE cancel_yn = '해지') AS n_cancel_rows,
        count(DISTINCT sha2_hash) AS n_customers,
        count(DISTINCT sha2_hash) FILTER (WHERE cancel_yn = '해지') AS n_churned_customers
    FROM churn_base
""").df()
  
truncate_check

,n_rows,n_cancel_rows,n_customers,n_churned_customers
0,2135261,150613,2135261,150613


In [10]:
baseline_trunc = duckdb.sql("""
    SELECT
        count(*) AS n_customers,
        count(*) FILTER (WHERE cancel_yn = '해지') AS n_churners,
        round(count(*) FILTER (WHERE cancel_yn = '해지') * 100.0 / count(*), 4) AS churn_rate_pct
    FROM churn_base
""").df()

baseline_trunc

,n_customers,n_churners,churn_rate_pct
0,2135261,150613,7.0536


In [11]:
categorical_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "AGMT_END_SEG", "BUNDLE_YN", "DIGITAL_GIGA_YN", "DIGITAL_ALOG_YN", "CH_LAST_DAYS_BF_GRP",
    "VOC_TOTAL_MONTH1_YN", "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM",
    "SMS_SEND_CLS_NM", "NFX_USE_YN", "YTB_USE_YN", "CH_FAV_RNK1",
]
cols_sql = ", ".join(categorical_cols)

baseline_rate_trunc = baseline_trunc['churn_rate_pct'][0]

categorical_churn_scan_trunc = duckdb.sql(f"""
    WITH base AS (
        SELECT
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_cancelled,
            {cols_sql}
        FROM churn_base
    ),
    unpvt AS (
        UNPIVOT base
        ON {cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        value,
        count(*) AS n,
        round(avg(is_cancelled) * 100, 4) AS churn_rate_pct,
        round(avg(is_cancelled) * 100 - {baseline_rate_trunc}, 4) AS deviation_from_baseline,
        round(avg(is_cancelled) * 100 / {baseline_rate_trunc}, 2) AS ratio_to_baseline
    FROM unpvt
    GROUP BY column_name, value
    HAVING count(*) >= 1000
    ORDER BY ratio_to_baseline DESC
""").df()

categorical_churn_scan_trunc.head(30)

,column_name,value,n,churn_rate_pct,deviation_from_baseline,ratio_to_baseline
0,AGMT_END_SEG,약정만료전 1개월,14096,24.6453,17.5917,3.49
1,AGMT_END_SEG,약정만료전 1~2개월,17777,22.5291,15.4755,3.19
2,AGMT_END_SEG,약정만료전 2~3개월,18948,19.6432,12.5896,2.78
3,AGE_GRP10,20대,20821,18.3036,11.2500,2.59
4,AGMT_END_SEG,약정만료전 3~6개월,68710,16.6555,9.6019,2.36
5,VOC_STOP_CANCEL_MONTH1_YN,Y,52430,15.8573,8.8037,2.25
6,SVC_USE_DAYS_GRP,24개월~ 36개월미만,216419,14.7510,7.6974,2.09
7,AGMT_END_SEG,약정만료전 6~9개월,69798,14.2855,7.2319,2.03
8,PAID_CHNL_CNT_GRP,3건 이상,2369,13.5078,6.4542,1.92
9,AGMT_END_SEG,약정만료후 1개월~2개월,8685,13.0109,5.9573,1.84


##### Top 5 Segments (One Value per Variable)
- From the scan above, take the highest value of each variable and rank them; this keeps one variable (e.g. contract expiry) from filling the whole list
- Two values are excluded: cancellation/suspension call (`VOC_STOP_CANCEL_MONTH1_YN = Y`) is an intent to cancel, not a trait; tenure 24–36 months overlaps with contract expiry (Part 4)

In [12]:
# Top 5: the highest value of each variable, ranked by ratio to the customer-level baseline
# Excluded because they are not customer traits to act on:
EXCLUDE = {
    "VOC_STOP_CANCEL_MONTH1_YN": "Y",           # cancellation/suspension call = intent to cancel
    "SVC_USE_DAYS_GRP": "24개월~ 36개월미만",     # overlaps with contract expiry (checked in Part 4)
}

keep = []
for _, row in categorical_churn_scan_trunc.iterrows():
    col = row['column_name']
    val = row['value']

    is_excluded = False
    if col in EXCLUDE:
        if EXCLUDE[col] == val:
            is_excluded = True

    if is_excluded:
        keep.append(False)
    else:
        keep.append(True)

candidates = categorical_churn_scan_trunc[keep]
candidates = candidates.sort_values('ratio_to_baseline', ascending=False)
top_per_variable = candidates.drop_duplicates(subset='column_name', keep='first')   # one value per variable

top5 = top_per_variable.head(5).reset_index(drop=True)
top5

,column_name,value,n,churn_rate_pct,deviation_from_baseline,ratio_to_baseline
0,AGMT_END_SEG,약정만료전 1개월,14096,24.6453,17.5917,3.49
1,AGE_GRP10,20대,20821,18.3036,11.2500,2.59
2,PAID_CHNL_CNT_GRP,3건 이상,2369,13.5078,6.4542,1.92
3,DIGITAL_GIGA_YN,Y,278824,12.9203,5.8667,1.83
4,CH_LAST_DAYS_BF_GRP,4주일전,7658,12.7579,5.7043,1.81


In [13]:
# Re-contracted customers: used in the Part 3 proposal (re-contract offers)
recontract = categorical_churn_scan_trunc[categorical_churn_scan_trunc['value'] == '재약정']
recontract

,column_name,value,n,churn_rate_pct,deviation_from_baseline,ratio_to_baseline
102,AGMT_KIND_NM,재약정,823673,4.3159,-2.7377,0.61


##### Before vs After
- Before: all rows (baseline 4.03%) / After: one row per customer (baseline 7.05%)
- The baselines differ, so compare **ratios to baseline**, not raw rates

In [14]:
# "Before" = same scan on all cleaned rows (Part 1 method), recomputed here so this
# cell doesn't depend on Part 1 having been run
churn_scan_before = duckdb.sql(f"""
    WITH base AS (
        SELECT
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_cancelled,
            {cols_sql}
        FROM {CLEAN}
    ),
    unpvt AS (
        UNPIVOT base
        ON {cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        value,
        round(avg(is_cancelled) * 100, 4) AS rate_before,
        round(avg(is_cancelled) * 100
              / (SELECT avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END) * 100 FROM {CLEAN}), 2) AS ratio_before
    FROM unpvt
    GROUP BY column_name, value
    HAVING count(*) >= 1000
""").df()

compare_trunc = (
    churn_scan_before
    .merge(
        categorical_churn_scan_trunc[['column_name', 'value', 'n', 'churn_rate_pct', 'ratio_to_baseline']]
        .rename(columns={'churn_rate_pct': 'rate_after', 'ratio_to_baseline': 'ratio_after'}),
        on=['column_name', 'value'],
    )
    .sort_values('ratio_after', ascending=False)
)

compare_trunc.head(30)

,column_name,value,rate_before,ratio_before,n,rate_after,ratio_after
65,AGMT_END_SEG,약정만료전 1개월,17.6548,4.38,14096,24.6453,3.49
63,AGMT_END_SEG,약정만료전 1~2개월,14.8886,3.69,17777,22.5291,3.19
45,AGMT_END_SEG,약정만료전 2~3개월,12.0250,2.98,18948,19.6432,2.78
66,AGE_GRP10,20대,11.7560,2.91,20821,18.3036,2.59
86,AGMT_END_SEG,약정만료전 3~6개월,8.2893,2.05,68710,16.6555,2.36
83,VOC_STOP_CANCEL_MONTH1_YN,Y,13.8674,3.44,52430,15.8573,2.25
24,SVC_USE_DAYS_GRP,24개월~ 36개월미만,7.8075,1.94,216419,14.7510,2.09
4,AGMT_END_SEG,약정만료전 6~9개월,4.4214,1.10,69798,14.2855,2.03
15,PAID_CHNL_CNT_GRP,3건 이상,7.7121,1.91,2369,13.5078,1.92
30,AGMT_END_SEG,약정만료후 1개월~2개월,9.4686,2.35,8685,13.0109,1.84


In [15]:
numeric_cols = [
    "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB", "DIGITAL_SCRB",
    "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "TV_I_CNT",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM",
    "KIDS_USE_PV_MONTH1", "INHOME_RATE",
]
avg_exprs = ""
for c in numeric_cols:
    avg_part = "avg(" + c + ")"
    round_part = "round(" + avg_part + ", 3)"
    expr = round_part + " AS avg_" + c
    if avg_exprs == "":
        avg_exprs = expr
    else:
        avg_exprs = avg_exprs + ", " + expr

numeric_churn_scan_trunc = duckdb.sql(f"""
    SELECT cancel_yn, count(*) AS n, {avg_exprs}
    FROM churn_base
    GROUP BY cancel_yn
""").df()

numeric_churn_scan_trunc.T

,0,1
cancel_yn,해지,유지
n,150613,1984648
avg_TOTAL_USED_DAYS,2183.926,2581.682
avg_TV_SCRB,1.713,1.839
avg_ANALOG_SCRB,0.016,0.022
avg_DIGITAL_SCRB,1.697,1.817
avg_TOTAL_INTERNET_SCRB,0.653,0.5
avg_GIGA_INTERNET_SCRB,0.246,0.126
avg_TV_I_CNT,2.366,2.338
avg_CH_HH_AVG_MONTH1,3.563,4.815


##### Findings
- 2,135,261 customers, 150,613 churners → customer-level churn rate **7.05%**
- Row-level counting overstated the risk right before contract expiry: 1 month before expiry 4.38x → **3.49x**
- Top 5 (`top5-01`, one value per variable): 1 month before contract expiry 3.49x, age 20s 2.59x, 3+ paid channels 1.92x, digital TV + giga internet bundle 1.83x, last viewing 4 weeks ago 1.81x
- Re-contracted customers (`recontract-01`): 0.61x, the lowest-risk contract type
- Excluded: cancellation/suspension call (2.25x) looks like a signal of intent rather than a cause; tenure 24–36 months (2.09x) overlaps with contract expiry (Part 4)
- Churners have shorter tenure (2,184 vs 2,582 days), lower viewing (3.56 vs 4.81) and `INHOME_RATE` (20.7 vs 24.7), and more giga internet lines (0.25 vs 0.13)

---

## Part 3. Where Do Churners Come From: Share by Contract Stage

##### Churner Share by Contract-Expiry Segment
- A high churn rate in a small segment still means few churners → look at **share of customers vs share of churners** together (customer level)

In [16]:
# Uses churn_base from the setup cell at the start of Part 2

# Chronological order: far before expiry -> expiry month -> long after expiry
EXPIRY_ORDER = [
    "약정만료전 12개월이상", "약정만료전 9~12개월", "약정만료전 6~9개월", "약정만료전 3~6개월",
    "약정만료전 2~3개월", "약정만료전 1~2개월", "약정만료전 1개월", "약정만료 1개월",
    "약정만료후 1개월~2개월", "약정만료후 2개월~3개월", "약정만료후 3~6개월",
    "약정만료후 6~9개월", "약정만료후 9~12개월", "약정만료후 12개월이상",
]

expiry_share = duckdb.sql(f"""
    WITH seg AS (
        SELECT
            AGMT_END_SEG,
            count(*) AS n,
            count(*) FILTER (WHERE cancel_yn = '해지') AS n_churn
        FROM churn_base
        GROUP BY AGMT_END_SEG
        HAVING count(*) >= 1000                 -- drops '기타' (3 customers)
    ),
    total AS (
        SELECT sum(n) AS total_n, sum(n_churn) AS total_churn FROM seg
    )
    SELECT
        AGMT_END_SEG,
        n,
        n_churn,
        round(n_churn * 100.0 / n, 4) AS churn_rate_pct,
        round(n * 100.0 / total_n, 2) AS share_of_customers_pct,
        round(n_churn * 100.0 / total_churn, 2) AS share_of_churners_pct,
        round((n_churn * 1.0 / total_churn) / (n * 1.0 / total_n), 2) AS concentration
    FROM seg, total
""").df()

order_number = {}
for i in range(len(EXPIRY_ORDER)):
    order_number[EXPIRY_ORDER[i]] = i

expiry_share['order'] = expiry_share['AGMT_END_SEG'].map(order_number)
expiry_share = expiry_share.sort_values('order')
expiry_share = expiry_share.drop(columns='order')
expiry_share = expiry_share.reset_index(drop=True)

expiry_share

,AGMT_END_SEG,n,n_churn,churn_rate_pct,share_of_customers_pct,share_of_churners_pct,concentration
0,약정만료전 12개월이상,980141,42187,4.3042,45.90,28.01,0.61
1,약정만료전 9~12개월,81648,6864,8.4068,3.82,4.56,1.19
2,약정만료전 6~9개월,69798,9971,14.2855,3.27,6.62,2.03
3,약정만료전 3~6개월,68710,11444,16.6555,3.22,7.60,2.36
4,약정만료전 2~3개월,18948,3722,19.6432,0.89,2.47,2.78
5,약정만료전 1~2개월,17777,4005,22.5291,0.83,2.66,3.19
6,약정만료전 1개월,14096,3474,24.6453,0.66,2.31,3.49
7,약정만료 1개월,29950,3746,12.5075,1.40,2.49,1.77
8,약정만료후 1개월~2개월,8685,1130,13.0109,0.41,0.75,1.84
9,약정만료후 2개월~3개월,9724,1208,12.4229,0.46,0.80,1.76


##### Sizing the Pre-Expiry Window
- Start from the segments closest to expiry and widen the window step by step (1–3, 1–6, 1–9, 1–12 months before expiry)
- Compare how many churners each window reaches against how efficient it is, to decide when retention should start

In [17]:
TARGET_WINDOWS = {
    "만료 1~3개월 전": ["약정만료전 1개월", "약정만료전 1~2개월", "약정만료전 2~3개월"],
    "만료 1~6개월 전": ["약정만료전 1개월", "약정만료전 1~2개월", "약정만료전 2~3개월", "약정만료전 3~6개월"],
    "만료 1~9개월 전": ["약정만료전 1개월", "약정만료전 1~2개월", "약정만료전 2~3개월", "약정만료전 3~6개월",
                    "약정만료전 6~9개월"],
    "만료 1~12개월 전": ["약정만료전 1개월", "약정만료전 1~2개월", "약정만료전 2~3개월", "약정만료전 3~6개월",
                     "약정만료전 6~9개월", "약정만료전 9~12개월"],
}

total_n = expiry_share['n'].sum()
total_churn = expiry_share['n_churn'].sum()

rows = []
for name, segs in TARGET_WINDOWS.items():
    sub = expiry_share[expiry_share['AGMT_END_SEG'].isin(segs)]
    n, churn = sub['n'].sum(), sub['n_churn'].sum()
    rows.append({
        "window": name,
        "n_customers": int(n),
        "n_churners": int(churn),
        "share_of_customers_pct": round(n * 100 / total_n, 2),
        "share_of_churners_pct": round(churn * 100 / total_churn, 2),
        "concentration": round((churn / total_churn) / (n / total_n), 2),
        "churn_rate_pct": round(churn * 100 / n, 2),
    })

target_windows = pd.DataFrame(rows)
target_windows

,window,n_customers,n_churners,share_of_customers_pct,share_of_churners_pct,concentration,churn_rate_pct
0,만료 1~3개월 전,50821,11201,2.38,7.44,3.12,22.04
1,만료 1~6개월 전,119531,22645,5.60,15.04,2.69,18.94
2,만료 1~9개월 전,189329,32616,8.87,21.66,2.44,17.23
3,만료 1~12개월 전,270977,39480,12.69,26.21,2.07,14.57


##### Findings & Proposal
**1. Choosing the retention window**
| Window | Share of customers | Share of churners | Concentration | Added segment alone |
|---|---|---|---|---|
| 1–3 months before expiry | 2.4% | 7.4% | 3.12 | 3.12 |
| 1–6 months before expiry | 5.6% | 15.0% | 2.69 | 2.36 (3–6 months) |
| 1–9 months before expiry | 8.9% | 21.7% | 2.44 | 2.03 (6–9 months) |
| 1–12 months before expiry | 12.7% | 26.2% | 2.07 | 1.19 (9–12 months) |

- Recommendation: start retention 6 months before expiry, contacting the 1–3 month segment first
  - 1–3 months: highest efficiency (3.1x), reaches 7.4% of churners
  - Adding 3–6 months: the added segment alone is still 2.4x, doubling reach to 15.0% at 2.7x overall
  - 9–12 months is close to average (1.2x), so stop before it
- With a smaller budget, keep only 1–3 months; with a larger one, extend to 9 months (21.7% at 2.4x). The final window should be set by contact capacity, offer cost and an experiment

**2. Response by contract stage**
| Contract stage | Share of customers | Share of churners | Action |
|---|---|---|---|
| 1–6 months before expiry | 5.6% | 15.0% | Retention offer from 6 months before expiry |
| 12+ months after expiry | 34.7% | 35.5% | Out of contract but still subscribed → re-contract offers (re-contracted customers: 0.61x, Part 2) |
| 12+ months before expiry | 45.9% | 28.0% | No contract trigger → early warning based on high-risk segments (Part 4) |

- The remaining ~21% falls between these stages (6–12 months before expiry, the expiry month, 1–12 months after expiry)
- Caveat: the churner share is an upper bound on what a retention offer could reach; the real effect needs an experiment to measure

---

## Part 4. Overlap Check: Compare Segments at the Same Contract Stage

##### Method
- Group customers with the same contract-expiry timing, then compare segments' churn rates within each group (customer level)
- If a segment is still higher under the same contract timing, its risk is independent of the contract
- Each cell also shows the number of customers and a 95% confidence interval, so small groups can be read with care

In [18]:
# Uses churn_base from the setup cell at the start of Part 2

EXPIRY_GROUP = """
    CASE
        WHEN AGMT_END_SEG IN ('약정만료전 1개월', '약정만료전 1~2개월', '약정만료전 2~3개월', '약정만료전 3~6개월')
            THEN '1) 만료 1~6개월 전'
        WHEN AGMT_END_SEG IN ('약정만료전 6~9개월', '약정만료전 9~12개월') THEN '2) 만료 6~12개월 전'
        WHEN AGMT_END_SEG = '약정만료전 12개월이상' THEN '3) 만료 12개월 이상 전'
        ELSE '4) 만료 이후'
    END"""

SEGMENTS = {
    "연령대": "CASE WHEN AGE_GRP10 = '20대' THEN '20대' WHEN AGE_GRP10 = '30대' THEN '30대' ELSE '그 외' END",
    "가입 기간": """CASE WHEN SVC_USE_DAYS_GRP = '24개월~ 36개월미만' THEN '24~36개월'
                         WHEN SVC_USE_DAYS_GRP = '6개월미만' THEN '6개월 미만' ELSE '그 외' END""",
    "유료 채널 가입 수": """CASE WHEN PAID_CHNL_CNT_GRP IN ('2건', '3건 이상') THEN '2건 이상'
                                WHEN PAID_CHNL_CNT_GRP = '1건' THEN '1건' ELSE '0건' END""",
    "최근 시청일": """CASE WHEN CH_LAST_DAYS_BF_GRP IN ('4주일전', '3개월내없음') THEN '4주 이상 시청 없음'
                          ELSE '그 외' END""",
    "디지털과 기가 결합": "CASE WHEN DIGITAL_GIGA_YN = 'Y' THEN '디지털과 기가 결합' ELSE '그 외' END",
}


def within_expiry(segment_sql):
    # 95% confidence interval (normal approximation): rate ± 1.96 × sqrt(rate × (1 − rate) / n)
    long = duckdb.sql(f"""
        WITH labeled AS (
            SELECT
                {EXPIRY_GROUP} AS expiry_group,
                {segment_sql} AS segment,
                CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_churn
            FROM churn_base
        )
        SELECT
            expiry_group,
            segment,
            count(*) AS n,
            round(avg(is_churn) * 100, 3) AS churn_rate_pct,
            round((avg(is_churn) - 1.96 * sqrt(avg(is_churn) * (1 - avg(is_churn)) / count(*))) * 100, 2) AS ci_low_pct,
            round((avg(is_churn) + 1.96 * sqrt(avg(is_churn) * (1 - avg(is_churn)) / count(*))) * 100, 2) AS ci_high_pct
        FROM labeled
        GROUP BY 1, 2
        ORDER BY 1, 2
    """).df()
    rates = {}
    for _, row in long.iterrows():
        seg = row['segment']
        grp = row['expiry_group']
        if seg not in rates:
            rates[seg] = {}
        rates[seg][grp] = row['churn_rate_pct']

    rates = pd.DataFrame(rates).T
    rates = rates[sorted(rates.columns)]
    rates = rates.sort_index()

    near = duckdb.sql(f"""
        SELECT
            {segment_sql} AS segment,
            round(avg(CASE WHEN {EXPIRY_GROUP} = '1) 만료 1~6개월 전' THEN 1 ELSE 0 END) * 100, 2)
                AS pct_near_expiry
        FROM churn_base
        GROUP BY 1
    """).df().set_index('segment')
    summary = rates.join(near)
    return summary, long


overlap_results = {}
overlap_details = {}
for name, sql in SEGMENTS.items():
    summary, detail = within_expiry(sql)
    overlap_results[name] = summary
    overlap_details[name] = detail

for name in overlap_results:
    print(f"\n[{name}]  churn rate (%) by expiry group, + share sitting 1~6 months before expiry")
    display(overlap_results[name])
    print(f"[{name}]  number of customers and 95% confidence interval for each cell")
    display(overlap_details[name])


[연령대]  churn rate (%) by expiry group, + share sitting 1~6 months before expiry


,1) 만료 1~6개월 전,2) 만료 6~12개월 전,3) 만료 12개월 이상 전,4) 만료 이후,pct_near_expiry
20대,39.974,23.956,14.846,20.154,7.43
30대,33.045,20.022,7.758,13.946,8.12
그 외,17.611,10.374,3.931,7.553,5.45


[연령대]  number of customers and 95% confidence interval for each cell


,expiry_group,segment,n,churn_rate_pct,ci_low_pct,ci_high_pct
0,1) 만료 1~6개월 전,20대,1546,39.974,37.53,42.42
1,1) 만료 1~6개월 전,30대,8089,33.045,32.02,34.07
2,1) 만료 1~6개월 전,그 외,109896,17.611,17.39,17.84
3,2) 만료 6~12개월 전,20대,1820,23.956,22.00,25.92
4,2) 만료 6~12개월 전,30대,9085,20.022,19.20,20.84
5,2) 만료 6~12개월 전,그 외,140541,10.374,10.21,10.53
6,3) 만료 12개월 이상 전,20대,14334,14.846,14.26,15.43
7,3) 만료 12개월 이상 전,30대,54729,7.758,7.53,7.98
8,3) 만료 12개월 이상 전,그 외,911078,3.931,3.89,3.97
9,4) 만료 이후,20대,3121,20.154,18.75,21.56



[가입 기간]  churn rate (%) by expiry group, + share sitting 1~6 months before expiry


...

,expiry_group,segment,n,churn_rate_pct,ci_low_pct,ci_high_pct
0,1) 만료 1~6개월 전,4주 이상 시청 없음,22758,32.165,31.56,32.77
1,1) 만료 1~6개월 전,그 외,96773,15.836,15.61,16.07
2,2) 만료 6~12개월 전,4주 이상 시청 없음,25113,17.768,17.29,18.24
3,2) 만료 6~12개월 전,그 외,126333,9.794,9.63,9.96
4,3) 만료 12개월 이상 전,4주 이상 시청 없음,138747,7.070,6.93,7.20
5,3) 만료 12개월 이상 전,그 외,841394,3.848,3.81,3.89
6,4) 만료 이후,4주 이상 시청 없음,171265,13.408,13.25,13.57
7,4) 만료 이후,그 외,712878,6.450,6.39,6.51



[디지털과 기가 결합]  churn rate (%) by expiry group, + share sitting 1~6 months before expiry


,1) 만료 1~6개월 전,2) 만료 6~12개월 전,3) 만료 12개월 이상 전,4) 만료 이후,pct_near_expiry
그 외,13.836,8.428,3.448,7.558,4.79
디지털과 기가 결합,33.848,22.177,7.807,15.677,10.94


[디지털과 기가 결합]  number of customers and 95% confidence interval for each cell


,expiry_group,segment,n,churn_rate_pct,ci_low_pct,ci_high_pct
0,1) 만료 1~6개월 전,그 외,89015,13.836,13.61,14.06
1,1) 만료 1~6개월 전,디지털과 기가 결합,30516,33.848,33.32,34.38
2,2) 만료 6~12개월 전,그 외,121839,8.428,8.27,8.58
3,2) 만료 6~12개월 전,디지털과 기가 결합,29607,22.177,21.70,22.65
4,3) 만료 12개월 이상 전,그 외,787561,3.448,3.41,3.49
5,3) 만료 12개월 이상 전,디지털과 기가 결합,192580,7.807,7.69,7.93
6,4) 만료 이후,그 외,858022,7.558,7.50,7.61
7,4) 만료 이후,디지털과 기가 결합,26121,15.677,15.24,16.12


##### Why '2+ Paid Channels' Instead of '3+'
- The Top list in Part 2 used `3건 이상` (3+ paid channels), but that group is small once split by contract stage
- Check each paid-channel group within each contract stage to see whether 2 and 3+ can be combined

In [19]:
# Uses churn_base (setup cell) and EXPIRY_GROUP (overlap-01)
# Why '2+ paid channels': compare 2 vs 3+ within each contract stage, with 95% confidence intervals
paid_grouping_check = duckdb.sql(f"""
    WITH labeled AS (
        SELECT
            {EXPIRY_GROUP} AS expiry_group,
            PAID_CHNL_CNT_GRP AS paid_channels,
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_churn
        FROM churn_base
    )
    SELECT
        expiry_group,
        paid_channels,
        count(*) AS n,
        round(avg(is_churn) * 100, 2) AS churn_rate_pct,
        round((avg(is_churn) - 1.96 * sqrt(avg(is_churn) * (1 - avg(is_churn)) / count(*))) * 100, 2) AS ci_low_pct,
        round((avg(is_churn) + 1.96 * sqrt(avg(is_churn) * (1 - avg(is_churn)) / count(*))) * 100, 2) AS ci_high_pct
    FROM labeled
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

paid_grouping_check

,expiry_group,paid_channels,n,churn_rate_pct,ci_low_pct,ci_high_pct
0,1) 만료 1~6개월 전,0건,114849,18.66,18.43,18.88
1,1) 만료 1~6개월 전,1건,4188,26.07,24.74,27.40
2,1) 만료 1~6개월 전,2건,397,25.69,21.39,29.99
3,1) 만료 1~6개월 전,3건 이상,97,22.68,14.35,31.01
4,2) 만료 6~12개월 전,0건,145061,10.97,10.81,11.13
5,2) 만료 6~12개월 전,1건,5523,14.27,13.35,15.19
6,2) 만료 6~12개월 전,2건,683,14.79,12.13,17.45
7,2) 만료 6~12개월 전,3건 이상,179,18.99,13.25,24.74
8,3) 만료 12개월 이상 전,0건,945252,4.19,4.15,4.23
9,3) 만료 12개월 이상 전,1건,30590,6.97,6.68,7.25


##### Findings
- 3+ alone is too small to judge: 1–6 months before expiry it has only 97 customers, and its interval (14.4–31.0%) overlaps 0 channels (18.4–18.9%)
- 2 and 3+ have overlapping intervals in every contract stage, so they behave alike → combined as '2+' (more customers, narrower intervals) for the overlap check and the risk signals

##### Why '4+ Weeks Without Viewing'
- `CH_LAST_DAYS_BF_GRP` has 6 values, from `일주일내` (watched within a week) to `3개월내없음` (no viewing in 3 months)
- The Top 5 picked `4주일전` (last viewing 4 weeks ago, 1.81x); check how large it is and whether it can be combined with its neighbour `3개월내없음`

In [20]:
# Uses churn_base (setup cell)
# Why '4+ weeks without viewing': churn rate by last-viewing group, from most recent to least recent
VIEW_ORDER = ["일주일내", "일주일전", "2주일전", "3주일전", "4주일전", "3개월내없음"]

order_number = {}
for i in range(len(VIEW_ORDER)):
    order_number[VIEW_ORDER[i]] = i

viewing_check = duckdb.sql("""
    SELECT
        CH_LAST_DAYS_BF_GRP AS last_viewing,
        count(*) AS n,
        round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS share_of_customers_pct,
        round(avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate_pct,
        round(avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END)
              / (SELECT avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END) FROM churn_base), 2) AS ratio_to_baseline
    FROM churn_base
    GROUP BY 1
""").df()

viewing_check['order'] = viewing_check['last_viewing'].map(order_number)
viewing_check = viewing_check.sort_values('order')
viewing_check = viewing_check.drop(columns='order')
viewing_check = viewing_check.reset_index(drop=True)

display(viewing_check)

# Combined signal used in Part 4: last viewing 4 weeks ago or no viewing in 3 months
viewing_combined = duckdb.sql("""
    SELECT
        count(*) AS n,
        round(count(*) * 100.0 / (SELECT count(*) FROM churn_base), 2) AS share_of_customers_pct,
        round(avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate_pct,
        round(avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END)
              / (SELECT avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END) FROM churn_base), 2) AS ratio_to_baseline
    FROM churn_base
    WHERE CH_LAST_DAYS_BF_GRP IN ('4주일전', '3개월내없음')
""").df()

viewing_combined

,last_viewing,n,share_of_customers_pct,churn_rate_pct,ratio_to_baseline
0,일주일내,1667787,78.11,5.70,0.81
1,일주일전,58428,2.74,9.72,1.38
2,2주일전,30898,1.45,10.25,1.45
3,3주일전,20265,0.95,10.93,1.55
4,4주일전,7658,0.36,12.76,1.81
5,3개월내없음,350225,16.40,12.44,1.76


,n,share_of_customers_pct,churn_rate_pct,ratio_to_baseline
0,357883,16.76,12.45,1.76


##### Findings
- The longer since last viewing, the higher the churn rate, step by step: within a week 5.70% → 1 week 9.72% → 2 weeks 10.25% → 3 weeks 10.93% → 4 weeks 12.76%
- `4주일전` is the Top 5 value but has only 7,658 customers (0.4%); `3개월내없음` is almost as high (12.44%) and covers 16.4% of customers
- There is no value between them (no 1–3 month group, as the source data is binned) → combined as **no viewing for 4+ weeks**: 357,883 customers, 1.76x
- Caveat: churners may have stopped viewing because they already cancelled, so this signal is the most exposed to the post-cancellation caveat

##### Findings
- **Higher in every contract group**: age 20s (e.g. 12+ months before expiry 14.85% vs 3.93%), paid-channel users, no viewing for 4+ weeks (12+ months before expiry 7.07% vs 3.85%), digital + giga bundle (12+ months before expiry 7.81% vs 3.45%), tenure under 6 months (12+ months before expiry 6.05% vs 3.84%)
- Tenure under 6 months is 0.92x overall only because almost all new customers sit 12+ months before expiry, the lowest-churn stage — within the same stage they are higher
- **Same signal as contract expiry**: tenure 24–36 months → 34% sit right before expiry, and with 12+ months to expiry there is no difference (3.70% vs 3.84%)
- Use: the four Top 5 traits (20s, 2+ paid channels, no viewing for 4+ weeks, digital + giga bundle) become the early-warning signals for in-contract customers (28% of churners); tenure under 6 months is also higher but is not in the Top 5, so it is reported here and not used as a signal
- **Not chance**: in every contract group, the 95% confidence interval of each high-risk segment sits above that of the rest (smallest cell: 2+ paid channels 1–6 months before expiry, 494 customers, 21.3–28.9% vs 18.4–18.9% for 0 channels)
- Caveats: age 20s is only 1% of customers; only contract timing is controlled

##### Stacking Risk Signals: Does Risk Rise When Signals Overlap?
- Each signal above was checked one at a time; an early-warning rule needs to know whether customers with several signals are riskier
- Signals = the Top 5 minus contract expiry (the stage itself is controlled): 20s, 2+ paid channels, no viewing for 4+ weeks, digital + giga bundle
- Count how many of the 4 signals each customer has (3 or more grouped as '3+')
- Compare churn rates by signal count within each contract stage, so contract timing stays controlled

In [21]:
# Uses churn_base (setup cell) and EXPIRY_GROUP (cell above)
# Top 5 minus contract expiry; small values combined with similar neighbours (paid-grouping-01, viewing-grouping-01)
SIGNALS = {
    "20대": "AGE_GRP10 = '20대'",
    "유료 채널 2건 이상": "PAID_CHNL_CNT_GRP IN ('2건', '3건 이상')",
    "4주 이상 시청 없음": "CH_LAST_DAYS_BF_GRP IN ('4주일전', '3개월내없음')",
    "디지털과 기가 결합": "DIGITAL_GIGA_YN = 'Y'",
}

signal_count = ""
for name, cond in SIGNALS.items():
    part = "CASE WHEN " + cond + " THEN 1 ELSE 0 END"
    if signal_count == "":
        signal_count = part
    else:
        signal_count = signal_count + " + " + part

signal_stack = duckdb.sql(f"""
    WITH scored AS (
        SELECT
            {EXPIRY_GROUP} AS expiry_group,
            least({signal_count}, 3) AS n_signals,          -- 3 or more grouped as '3+'
            cancel_yn
        FROM churn_base
    )
    SELECT
        expiry_group,
        CASE WHEN n_signals = 3 THEN '3+' ELSE CAST(n_signals AS VARCHAR) END AS n_signals,
        count(*) AS n_customers,
        count(*) FILTER (WHERE cancel_yn = '해지') AS n_churners,
        round(count(*) FILTER (WHERE cancel_yn = '해지') * 100.0 / count(*), 2) AS churn_rate_pct,
        round(count(*) * 100.0 / sum(count(*)) OVER (PARTITION BY expiry_group), 2) AS share_of_stage_customers_pct,
        round(count(*) FILTER (WHERE cancel_yn = '해지') * 100.0
              / sum(count(*) FILTER (WHERE cancel_yn = '해지')) OVER (PARTITION BY expiry_group), 2)
            AS share_of_stage_churners_pct
    FROM scored
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

signal_stack

,expiry_group,n_signals,n_customers,n_churners,churn_rate_pct,share_of_stage_customers_pct,share_of_stage_churners_pct
0,1) 만료 1~6개월 전,0,71025,7350,10.35,59.42,32.46
1,1) 만료 1~6개월 전,1,42119,12422,29.49,35.24,54.86
2,1) 만료 1~6개월 전,2,5973,2654,44.43,5.00,11.72
3,1) 만료 1~6개월 전,3+,414,219,52.90,0.35,0.97
4,2) 만료 6~12개월 전,0,100424,7037,7.01,66.31,41.80
5,2) 만료 6~12개월 전,1,45169,8158,18.06,29.83,48.46
6,2) 만료 6~12개월 전,2,5338,1479,27.71,3.52,8.79
7,2) 만료 6~12개월 전,3+,515,161,31.26,0.34,0.96
8,3) 만료 12개월 이상 전,0,668446,20273,3.03,68.20,48.06
9,3) 만료 12개월 이상 전,1,277974,17427,6.27,28.36,41.31


In [22]:
# Early-warning rule: customers with 2+ signals vs customers with no signal, by contract stage
# (uses signal_count and EXPIRY_GROUP from the cells above)
signal_rule = duckdb.sql(f"""
    WITH scored AS (
        SELECT
            {EXPIRY_GROUP} AS expiry_group,
            {signal_count} AS n_signals,
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_churn
        FROM churn_base
    )
    SELECT
        expiry_group,
        round(count(*) FILTER (WHERE n_signals >= 2) * 100.0 / count(*), 2) AS share_of_stage_customers_pct,
        round(sum(is_churn) FILTER (WHERE n_signals >= 2) * 100.0 / sum(is_churn), 2) AS share_of_stage_churners_pct,
        round(avg(is_churn) FILTER (WHERE n_signals >= 2) * 100, 2) AS churn_rate_2plus_pct,
        round(avg(is_churn) FILTER (WHERE n_signals = 0) * 100, 2) AS churn_rate_0_pct,
        round(avg(is_churn) FILTER (WHERE n_signals >= 2) / avg(is_churn) FILTER (WHERE n_signals = 0), 2) AS ratio_2plus_vs_0
    FROM scored
    GROUP BY 1
    ORDER BY 1
""").df()

signal_rule

,expiry_group,share_of_stage_customers_pct,share_of_stage_churners_pct,churn_rate_2plus_pct,churn_rate_0_pct,ratio_2plus_vs_0
0,1) 만료 1~6개월 전,5.34,12.69,44.98,10.35,4.35
1,2) 만료 6~12개월 전,3.86,9.74,28.02,7.01,4.00
2,3) 만료 12개월 이상 전,3.44,10.64,13.31,3.03,4.39
3,4) 만료 이후,0.71,2.16,23.64,6.16,3.84


##### Findings
- **Risk rises step by step as signals overlap, in every contract stage**
  - 12+ months before expiry (in contract): 0 signals 3.03% → 1: 6.27% → 2: 11.91% → 3+: 22.39%
  - 1–6 months before expiry: 0 signals 10.35% → 1: 29.49% → 2: 44.43% → 3+: 52.90%
- **Early-warning rule for in-contract customers** (`signal-rule-01`): 2+ signals → 3.44% of in-contract customers hold 10.64% of in-contract churners (about 3.1x), with a churn rate of 13.31%, 4.39x that of customers with no signal (3.03%)
- Near expiry, customers with signals can be contacted first within the 6-month retention window
- Caveats: signals are counted with equal weight; these are characteristics of churners, not causes

---

## Part 5. Data Limitation: Churners Are Cancelled From the First Month They Appear

##### Problem Identification
- Grouping churners by the month their `해지` first appears puts 96% of them in February 2023, the first month of the data
- Question: do churners ever appear as `유지` before turning `해지`? If not, the data never shows the moment of cancellation

In [23]:
import duckdb
import pandas as pd

CLEAN = "read_parquet('df_clean_part*.parquet')"

# Per customer: first month seen, first 해지 month, and number of 유지 rows
churn_timing_check = duckdb.sql(f"""
    WITH per_customer AS (
        SELECT
            sha2_hash,
            min(p_mt) AS first_seen,
            min(p_mt) FILTER (WHERE cancel_yn = '해지') AS first_cancel,
            count(*) FILTER (WHERE cancel_yn = '유지') AS n_active_rows
        FROM {CLEAN}
        GROUP BY sha2_hash
    )
    SELECT
        count(*) FILTER (WHERE first_cancel IS NOT NULL) AS n_churners,
        count(*) FILTER (WHERE first_cancel = first_seen) AS cancelled_in_first_month_seen,
        count(*) FILTER (WHERE first_cancel > first_seen) AS active_before_cancel,
        count(*) FILTER (WHERE first_cancel IS NOT NULL AND n_active_rows > 0) AS churners_with_any_active_row,
        count(*) FILTER (WHERE first_cancel IS NOT NULL AND first_seen = '202302') AS churners_first_seen_feb,
        count(*) FILTER (WHERE first_cancel IS NOT NULL AND first_seen > '202302') AS churners_first_seen_later
    FROM per_customer
""").df()

churn_timing_check.T

,0
n_churners,150613
cancelled_in_first_month_seen,150613
active_before_cancel,0
churners_with_any_active_row,0
churners_first_seen_feb,144728
churners_first_seen_later,5885


##### Findings
- All 150,613 churners are `해지` from the first month they appear, and **none has a single `유지` row** — no customer is ever observed switching from active to cancelled
- So `해지` behaves like a fixed customer label, not an event in time. The 96% in February is simply the churners already present when the data starts (144,728); the rest (5,885) first appear later
- Implications:
  - Churn cannot be predicted from this data: there is no "before cancellation" behavior to learn from
  - A monthly churn rate cannot be measured; the meaningful measure is the customer-level churn rate (7.05%)
  - Churners' attributes may describe their state at or after cancellation, so Parts 2–4 compare **who churners are**, not what caused the churn